In [ ]:
""" 
how to reduce memory usage in Pandas
内存优化 和 dtype 类型 管理
股价默认：float64，优化为float32，优化百份之五十
成交量：int64，优化为int32，优化百份之五十到百份之七十五
股票代码（重复）：object，优化为category，能优化近百份之九十的内存

对pandas的内存进行管理有两种方法，一种是只取需要的数据（某些特定的行或列，），另一种是优化DataFrame或Series中
数据的类型。
.memory_usage() 用来看运行用了多少内存
parquet 相对于.csv的巨大优化
parquet 的优势，更小，更快，且保留了dtype的信息

"""

In [ ]:
""" 
import pandas as pd
import numpy as np
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))

    return df
"""

In [ ]:
"""
下面这段代码，对比上面，优化了的程序是，知道是正数，然后使用uint来进行内存优化
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype
    if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.uint8).min and c_max < np.iinfo(np.uint8).max:
                    df[col] = df[col].astype(np.uint8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.uint16).min and c_max < np.iinfo(np.uint16).max:
                    df[col] = df[col].astype(np.uint16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.uint32).min and c_max < np.iinfo(np.uint32).max:
                    df[col] = df[col].astype(np.uint32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
                elif c_min > np.iinfo(np.uint64).min and c_max < np.iinfo(np.uint64).max:
                    df[col] = df[col].astype(np.uint64)
            elif str(col_type)[:5] == 'float':
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df
"""

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import os
import time

work_dir = Path.home() / 'Desktop' / '代码' / 'data analysis' / '新征程'
os.chdir(work_dir)  # 强制切换工作目录
print(f"工作目录: {os.getcwd()}")

from processor import DataProcessor

# ============================================================
# 设置工作目录（确保文件写入到正确位置）
# ============================================================
# 初始化
processor = DataProcessor(str(work_dir / 'data'))
processor.load_raw()

# ============================================================
# 任务 5-7 的代码

# ============================================================
# 【任务 5】诊断内存占用
# ============================================================

print("=== 任务 5：内存诊断 ===")
raw_df = processor.raw_data  # 现在 processor 已定义

# 全面内存诊断
print(raw_df.info(memory_usage='deep'))

# 详细内存分析
print(f"\n总内存占用: {raw_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n每列内存占用:")
print(raw_df.memory_usage(deep=True))

# ============================================================
# 【任务 6】优化 dtype
# ============================================================

print("\n=== 任务 6：dtype 优化 ===")

# 记录优化前的内存
mem_before = raw_df.memory_usage(deep=True).sum() / 1024**2

# 创建优化副本
df_optimized = raw_df.copy()

# 1. ticker → category
df_optimized['ticker'] = df_optimized['ticker'].astype('category')

# 2. 价格列 → float32
price_cols = ['Close', 'High', 'Low', 'Open']
for col in price_cols:
    if col in df_optimized.columns:
        df_optimized[col] = df_optimized[col].astype('float32')

# 3. 成交量 → int32
if 'Volume' in df_optimized.columns:
    df_optimized['Volume'] = df_optimized['Volume'].fillna(0).astype('int32')

# 记录优化后的内存
mem_after = df_optimized.memory_usage(deep=True).sum() / 1024**2

print(f"优化前内存: {mem_before:.2f} MB")
print(f"优化后内存: {mem_after:.2f} MB")
print(f"节省: {mem_before - mem_after:.2f} MB ({(1 - mem_after/mem_before)*100:.1f}%)")
print(f"\n优化后的 dtypes:")
print(df_optimized.dtypes)
# ============================================================

def auto_optimize_dtypes(df):
    """
    自动识别并优化 DataFrame 的 dtype
    规则：
    - 字符串列且唯一值比例 < 50% → category
    - float64 → float32
    - 整数列 >= 0 → uint32
    - 整数列有正有负 → int32
    """
    df_opt = df.copy()
    n_rows = len(df_opt)
    
    for col in df_opt.columns:
        col_data = df_opt[col]
        
        # 跳过日期列
        if pd.api.types.is_datetime64_any_dtype(col_data):
            continue
        
        # 字符串列 → category（唯一值比例 < 50%）
        if col_data.dtype == 'object':
            n_unique = col_data.nunique()
            if n_unique / n_rows < 0.5:
                df_opt[col] = col_data.astype('category')
        
        # float64 → float32
        elif col_data.dtype == 'float64':
            df_opt[col] = col_data.astype('float32')
        
        # int64 → int32 或 uint32
        elif col_data.dtype == 'int64':
            if col_data.min() >= 0:
                df_opt[col] = col_data.astype('uint32')
            else:
                df_opt[col] = col_data.astype('int32')
    
    return df_opt

# 测试自动优化
print("\n=== 任务 7：自动优化 ===")
df_auto = auto_optimize_dtypes(raw_df)
mem_auto = df_auto.memory_usage(deep=True).sum() / 1024**2

print(f"原始内存: {mem_before:.2f} MB")
print(f"自动优化后: {mem_auto:.2f} MB")
print(f"节省: {(1 - mem_auto/mem_before)*100:.1f}%")
print(f"\n优化后的 dtypes:")
print(df_auto.dtypes)


# ... 省略中间代码 ...

df_optimized = raw_df.copy()
# ... 优化代码 ...

# ============================================================
# 【任务 8】Parquet vs CSV 性能对比（使用绝对路径）
# ============================================================

print("\n=== 任务 8：Parquet vs CSV 性能对比 ===")

test_df = df_optimized

# 关键：使用绝对路径
csv_path = str(work_dir / 'test_data.csv')
parquet_path = str(work_dir / 'test_data.parquet')

print(f"CSV 路径: {csv_path}")
print(f"Parquet 路径: {parquet_path}")

# CSV 写入
start = time.time()
test_df.to_csv(csv_path, index=False)
csv_write_time = time.time() - start

# Parquet 写入
start = time.time()
test_df.to_parquet(parquet_path, index=False)
parquet_write_time = time.time() - start

# CSV 读取
start = time.time()
_ = pd.read_csv(csv_path)
csv_read_time = time.time() - start

# Parquet 读取
start = time.time()
_ = pd.read_parquet(parquet_path)
parquet_read_time = time.time() - start

# 文件大小
csv_size = os.path.getsize(csv_path) / 1024**2
parquet_size = os.path.getsize(parquet_path) / 1024**2

# 结果输出
print(f"\n{'指标':<20} {'CSV':>12} {'Parquet':>12} {'提升':>12}")
print("-" * 56)
print(f"{'写入时间 (秒)':<20} {csv_write_time:>12.4f} {parquet_write_time:>12.4f} {csv_write_time/parquet_write_time:>11.1f}x")
print(f"{'读取时间 (秒)':<20} {csv_read_time:>12.4f} {parquet_read_time:>12.4f} {csv_read_time/parquet_read_time:>11.1f}x")
print(f"{'文件大小 (MB)':<20} {csv_size:>12.2f} {parquet_size:>12.2f} {csv_size/parquet_size:>11.1f}x")

# 清理
os.remove(csv_path)
os.remove(parquet_path)

print("\n✅ 所有任务完成！")

In [ ]:
.info(memory_usage = deep)